# Olist Dataset
### This is a Brazilian ecommerce public dataset of orders made at Olist Store.

The dataset has information of 100k orders from 2016 to 2018 made at multiple marketplaces in Brazil.


In [ ]:
DATA = "../../data/v1/"

from IPython.display import Image, display
display(Image(filename=DATA + "dataset_relations.png", width=800))


# Dzien 2 / Krok 1: Eksploracja i scalanie danych Olist

**Cel:** zaladowac 7 tabel, zrozumiec co w nich siedzi, polaczyc w plaska ramke.

**Output:** `checkpoints/01_merged.parquet`

```
orders  ──┐
items   ──┤
payments──┤──► merge ──► 01_merged.parquet
reviews ──┤
customers─┤
products ─┤
sellers  ─┘
```


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


## Tabela orders

In [ ]:
# TODO: Zaladuj tabelę orders z pliku olist_orders_dataset.csv
# Zwroc uwage na kolumny z datami - powinny miec typ datetime64, nie string.
# Kolumny-timestampy: order_purchase_timestamp, order_approved_at,
#                    order_delivered_carrier_date, order_delivered_customer_date,
#                    order_estimated_delivery_date
# Wskazowka: pd.read_csv(..., parse_dates=[...])

orders = ___

print(f"orders: {orders.shape}")
print(orders["order_status"].value_counts().to_string())


## Tabele items i payments

In [ ]:
# TODO: Zaladuj items (olist_order_items_dataset.csv) i payments (olist_order_payments_dataset.csv)
# items ma kolumne shipping_limit_date - tez jako datetime
# Wyswietl shape i rozklad payment_type

items    = ___
payments = ___

print(f"items:    {items.shape}")
print(f"payments: {payments.shape}")
print()
print(payments["payment_type"].value_counts().to_string())


In [ ]:
items_per_order = items.groupby("order_id")["order_item_id"].count()
print(f"Produkty na zamowienie: avg={items_per_order.mean():.2f}, max={items_per_order.max()}")
print(f"{(items_per_order > 1).mean():.1%} zamowien ma wiecej niz 1 produkt")


## Tabela reviews i rozklad ocen

In [ ]:
# TODO: Zaladuj reviews (olist_order_reviews_dataset.csv)
# Narysuj histogram rozkladu review_score (1-5)
# Wskazowka: go.Bar z kolorami per ocena, lub px.histogram

reviews = ___

# TODO: wykres rozkladu ocen


## Pozostale tabele

In [ ]:
# TODO: Zaladuj customers, products, categories (product_category_name_translation.csv), sellers
# Wyswietl shape kazdej tabeli

customers  = ___
products   = ___
categories = ___
sellers    = ___

for name, df in [("customers",customers),("products",products),("sellers",sellers)]:
    print(f"  {name:10s}: {df.shape[0]:>6,} rows x {df.shape[1]} cols")


In [ ]:
# TODO: Narysuj top 15 kategorii produktow wg liczby produktow
# Wskazowka: df.value_counts()


## Agregacje: jeden wiersz per order_id

Items i payments maja relacje N:1 z orders.
Przed mergem agregujemy do jednego wiersza per zamowienie.

Analogia do SQL:
```sql
SELECT order_id,
       SUM(price) as price,
       COUNT(order_item_id) as items_count,
       ...
FROM items GROUP BY order_id
```


In [ ]:
# TODO: Zagreguj items do jednego wiersza per order_id
# Potrzebne: price (sum), freight_value (sum), items_count (count z order_item_id),
#            product_id (first), seller_id (first)
items_agg = items.groupby("order_id").agg(
    # TODO
).reset_index()

# TODO: Zagreguj payments - payment_type (first), payment_installments (max), payment_value (sum)
payments_agg = payments.groupby("order_id").agg(
    # TODO
).reset_index()

reviews_clean = reviews[["order_id","review_score"]].drop_duplicates("order_id")

print(f"items_agg:    {items_agg.shape[0]:,} rows")
print(f"payments_agg: {payments_agg.shape[0]:,} rows")


## Merge: laczenie 7 tabel

Odpowiednik JOIN w SQL. `how="left"` zachowuje wszystkie wiersze z lewej tabeli.


In [ ]:
# TODO: Polacz wszystkie tabele w jedna plaska ramke
# orders -> items_agg (order_id) -> payments_agg (order_id) -> reviews_clean (order_id)
#        -> customers (customer_id) -> products[wybrane kolumny] (product_id)
#        -> categories (product_category_name) -> sellers[seller_id,seller_state] (seller_id)
# Wskazowka: df.merge(inna_df, on="kolumna", how="left")

df = orders.merge(...)
    

In [ ]:
# TODO: Odfiltruj tylko dostarczone zamowienia z recenzja
# Warunki: order_status == "delivered" i nie-null review_score
# Wskazowka: .query() lub boolean indexing, potem .dropna(subset=[...]).copy()

df = ___

# TODO: Stworz kolumne is_bad_experience = 1 gdy review_score == 1, inaczej 0
df["is_bad_experience"] = ___

neg_rate = df["is_bad_experience"].mean()
print(f"Finalny dataset: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Bad experience (1-gwiazdka): {neg_rate:.1%}")
print(f"scale_pos_weight dla XGBoost: {(1-neg_rate)/neg_rate:.1f}")


In [ ]:
# Wizualizacja rozkladu targetu
counts = df["is_bad_experience"].value_counts().sort_index()
fig = go.Figure(go.Bar(
    x=["Dobra recenzja (0)", "Zla recenzja (1)"],
    y=counts.values,
    text=[f"{v:,} ({v/len(df)*100:.1f}%)" for v in counts.values],
    textposition="outside",
))
fig.update_layout(title="Rozklad targetu is_bad_experience", height=380)
fig.show()


## EDA: Analiza scalonego zbioru

Dopiero po scaleniu mozemy laczyc informacje z roznych tabel:
"Czy opoznienie dostawy wiaze sie ze zlymi recenzjami?" wymaga danych z `orders` i `reviews` jednoczesnie.

**Zbadaj przynajmniej:**
1. Jak opoznienie dostawy (`order_delivered_customer_date - order_estimated_delivery_date`) rozklada sie dla dobrych i zlych recenzji?
2. Ktore kategorie produktow maja najwyzszy odsetek zlych recenzji (min. 200 zamowien)?
3. Jedno wlasne pytanie - co Cie ciekawi w tym datasecie?

Sciaga: `notebooks/building_blocks_and_examples/eda_feature_engineering.ipynb`


In [ ]:
# TODO: EDA - przeprowadz analize wg powyzszych pytan
# Wzoruj sie na: notebooks/building_blocks_and_examples/eda_feature_engineering.ipynb
# Do dyspozycji: go.Histogram, px.bar, groupby + agg, .corr()

# 1. Opoznienie dostawy vs ocena
df_eda = df.copy()
df_eda["delivery_delay_days"] = ___  # TODO

# TODO: narysuj histogram opoznien osobno dla is_bad_experience=0 i =1


In [ ]:
# TODO: 2. Kategorie z najwyzszym bad-experience rate (min 200 zamowien)


In [ ]:
# TODO: 3. Inwencja własna - co warto zbadac przed budowaniem cech?


## Checkpoint: zapisz scalony dataset

Zapisujemy do Parquet, nie CSV.

**Dlaczego Parquet a nie CSV?**

| Cecha | CSV | Parquet |
|---|---|---|
| Typy danych | wszystko string, gubisz datetime | typy zachowane (datetime64, int, float) |
| Rozmiar | pelny tekst | kolumnowy + kompresja, ~5-10x mniejszy |
| Wczytywanie | caly plik | tylko potrzebne kolumny |
| Null vs pusty string | nie rozroznia | rozroznia (prawdziwy NaN) |


In [ ]:
CHECKPOINT = "checkpoints/01_merged.parquet"
df.to_parquet(CHECKPOINT, index=False)
print(f"Zapisano: {CHECKPOINT}")
print(f"Shape: {df.shape}")
